In [8]:
from pydantic import BaseModel
from pathlib import Path


class AzureConfig(BaseModel):
    subscription_id: str
    resource_group: str
    function_app_name: str


config = AzureConfig.model_validate_json(Path("azure_config.json").open().read())

In [9]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

In [10]:
from azure.mgmt.eventgrid import EventGridManagementClient


eventgrid_client = EventGridManagementClient(
    credential=credential,
    subscription_id=config.subscription_id,
)

In [11]:
topic_name = "TestiTopic"

# Check if topic exists

In [12]:
from azure.core.exceptions import ResourceNotFoundError


def topic_exists(resource_group_name: str, name: str) -> bool:
    try:
        eventgrid_client.topics.get(
            resource_group_name=resource_group_name,
            topic_name=name,
        )
        return True
    except ResourceNotFoundError:
        return False


exists = topic_exists(config.resource_group, topic_name)
print(f"Topic '{topic_name}' exists: {exists}")

Topic 'TestiTopic' exists: True


# Create topic

In [13]:
from azure.mgmt.eventgrid.models import Topic

location = "germanywestcentral"  # adjust to match your resource group location

if not exists:
    poller = eventgrid_client.topics.begin_create_or_update(
        resource_group_name=config.resource_group,
        topic_name=topic_name,
        topic_info=Topic(location=location),
    )
    created_topic = poller.result()
    print(f"Topic '{created_topic.name}' created at {created_topic.endpoint}")
else:
    print(f"Topic '{topic_name}' already exists, skipping creation")

Topic 'TestiTopic' already exists, skipping creation


# Create subscription to Azure Function

In [22]:
function_name = "create_calendar_event"
subscription_name = f"CreateCalendarEvent"

## Check if subscription exists

In [23]:
topic_scope = (
    f"/subscriptions/{config.subscription_id}"
    f"/resourceGroups/{config.resource_group}"
    f"/providers/Microsoft.EventGrid/topics/{topic_name}"
)


def subscription_exists(scope: str, name: str) -> bool:
    try:
        eventgrid_client.event_subscriptions.get(
            scope=scope,
            event_subscription_name=name,
        )
        return True
    except ResourceNotFoundError:
        return False


sub_exists = subscription_exists(topic_scope, subscription_name)
print(f"Subscription '{subscription_name}' exists: {sub_exists}")

Subscription 'CreateCalendarEvent' exists: False


## Create subscription

In [25]:
config

AzureConfig(subscription_id='fcdc0c79-2df2-4d19-8f7e-607a4fa79363', resource_group='Sebastian', function_app_name='Sebastian')

In [26]:
from azure.mgmt.eventgrid.models import (
    EventSubscription,
    AzureFunctionEventSubscriptionDestination,
)

function_resource_id = (
    f"/subscriptions/{config.subscription_id}"
    f"/resourceGroups/Sebastian_group"
    f"/providers/Microsoft.Web/sites/{config.function_app_name}"
    f"/functions/{function_name}"
)

if not sub_exists:
    destination = AzureFunctionEventSubscriptionDestination(
        resource_id=function_resource_id,
    )
    poller = eventgrid_client.event_subscriptions.begin_create_or_update(
        scope=topic_scope,
        event_subscription_name=subscription_name,
        event_subscription_info=EventSubscription(destination=destination),
    )
    created_subscription = poller.result()
    print(f"Subscription '{created_subscription.name}' created")
else:
    print(f"Subscription '{subscription_name}' already exists, skipping creation")

Subscription 'CreateCalendarEvent' created


# Get topic endpoint and key

In [ ]:
def get_topic_info(resource_group_name: str, topic_name: str):
    topic = eventgrid_client.topics.get(
        resource_group_name=resource_group_name,
        topic_name=topic_name,
    )

    keys = eventgrid_client.topics.list_shared_access_keys(
        resource_group_name=resource_group_name,
        topic_name=topic_name,
    )
    return topic.endpoint, keys.key1


endpoint, key = get_topic_info(config.resource_group, topic_name)
print(f"{endpoint=}")
# print(f"{key=}")

endpoint='https://testitopic.germanywestcentral-1.eventgrid.azure.net/api/events'
